In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from f1.data import load_tables
from f1.dnf import classify_dnf
from f1.features import (
    build_master,
    cluster_drivers,
    driver_performance_table,
    pace_consistency,
)
from f1.viz import plot_matrix, plot_metronome, plot_reliability

sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format
plt.rcParams["figure.dpi"] = 110

print("✅ Environment setup complete.")

1. Data Loading

In [ ]:
# Reads ./data, $F1_DATA_DIR, or downloads the Kaggle dataset via kagglehub.
tables = load_tables()
print(f"✅ Data loaded. Lap times records: {len(tables['lap_times']):,}")

2. Master Data Preparation

In [ ]:
# Era-neutral performance: finish percentile (1 = win, 0 = last) instead of raw points,
# which are not comparable across scoring systems.
master = build_master(tables["results"], tables["drivers"], tables["status"])
master["DNF_Reason"] = master["status"].apply(classify_dnf)

print("✅ Master dataframe created.")
master.head(3)

3. Analysis: Driver Profiling (Clustering)

In [ ]:
# Experienced drivers only (>20 races), then K-Means on robust-scaled features.
driver_stats = cluster_drivers(driver_performance_table(master), k=4, seed=42)

print("✅ Clustering complete. Drivers categorized.")
driver_stats["Category"].value_counts()

4. Analysis: Consistency & Reliability

In [ ]:
# Pace relative to the race median (laps > 107% of median dropped), std per driver-race.
drivers = tables["drivers"]
laps = tables["lap_times"].merge(drivers[["driverId", "forename", "surname"]], on="driverId")
laps["driver_name"] = laps["forename"] + " " + laps["surname"]

consistency = pace_consistency(laps)
# Lap times exist only from 1996, so this keeps modern drivers only.
full_stats = driver_stats.merge(consistency, on="Driver Name", how="inner")

print(f"✅ Consistency computed for {len(full_stats)} drivers.")

5. Visualization: Consistency & Reliability

In [ ]:
plot_metronome(full_stats, "images/human-metronome.png")
plot_reliability(master, driver_stats, "images/reliability-analysis.png")

6. Visualization: Performance Matrix

In [ ]:
plot_matrix(full_stats, "images/performance-matrix.png")

7. Validating the Number of Clusters

In [ ]:
from f1.clustering import (
    compare_algorithms,
    evaluate_k,
    plot_k_selection,
    plot_pca,
    scale_features,
)

X = scale_features(driver_stats)
k_table = evaluate_k(X, ks=range(2, 11))
plot_k_selection(k_table, "images/k-selection.png")
k_table.round(3)

In [ ]:
# K-Means vs GMM vs HDBSCAN at k=4. HDBSCAN labels most drivers as noise:
# drivers form a continuum, not density-separated groups.
display(compare_algorithms(X, k=4).round(3))
plot_pca(X, driver_stats["cluster_id"].to_numpy(), "images/clusters-pca.png")

**Reading the result.** Silhouette and Davies-Bouldin are best at k=2 (backmarkers vs everyone else), which is too coarse to be useful. From k=3 up, k=4 is the finest split that keeps silhouette above 0.5 (0.54) and near-perfect seed stability (ARI = 0.98), while k=5 drops to 0.43 and 0.56, and the elbow flattens after it. GMM scores lower and HDBSCAN leaves 66 of 246 drivers (27%) unassigned, so K-Means with k=4 is kept. The tiers are a convenient segmentation of a continuum, not natural groups.

8. Criteria Profile: Racecraft, Reliability and Teammate-Relative

In [ ]:
from f1.profile import (
    build_profile,
    plot_correlation,
    redundancy,
)

profile = build_profile(master)
corr, explained = redundancy(profile)
plot_correlation(corr, "images/criteria-correlation.png")
print("PCA explained variance:", explained.round(3), "| cumulative:", explained.cumsum().round(3))

In [ ]:
# Teammate delta: finish percentile minus the other driver(s) in the same car.
# Needs >= 20 shared races, so it separates driver from car better than raw results.
shown = profile.dropna(subset=["Teammate Delta"]).sort_values("Teammate Delta", ascending=False)
cols = ["Driver Name", "Teammate Delta", "Teammate Races", "Win Rate"]
display(shown[cols].head(10).round(3))
display(shown[cols].tail(5).round(3))

**Reading the result.** Win rate, podium rate and finish percentile are highly redundant (r = 0.63 to 0.86), and the first principal component explains 69% of the variance, so they measure mostly the same thing. Positions gained is *negatively* correlated with podium rate (r = -0.51): front-runners start near the front and have little to gain, so it measures starting position as much as racecraft. Mechanical DNF rate is almost independent of performance (|r| < 0.1), a clean "car luck" axis. Teammate delta correlates only moderately with results (r = 0.43 to 0.53), so it adds car-adjusted information. Caveat: when a car is dominant both teammates finish near the top, which compresses the delta (Hamilton ranks 58th of 251), so a low delta is not proof of a weaker driver.